In [1]:
import os, sys, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ["TORCH_NVML_DISABLED"] = "1"
torch.cuda.empty_cache()


os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
import argparse


In [2]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)
ds = VQADataset(config)
df = ds.load_df()


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Post-edit predictions will be saved to results/pred_postedit/ft/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', pred_postedit_dir='results/pred_postedit/ft/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.language_model.layers.16.mlp.gate_proj.weight'], processor_class=None, tokenizer_class=None, temperature=0.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', 

In [3]:
df

,uid,image_path,question,answer,rationale,choices,idx_choices,image_info_source,image_info_split,image_info_id,cot
0,1,data/images/fvqa/ILSVRC2012_test_00000444.JPEG,Tell me the name of the cosmetics shown in thi...,lipstick,lipstick belongs to the category of Cosmetics.,foundation; eyeshadow; lipstick; mascara,(A) foundation\n(B) eyeshadow\n(C) lipstick\n(...,ILSVRC,test,test_444,The image shows a lipstick. Lipsticks are cyli...
1,2,data/images/fvqa/ILSVRC2012_test_00000444.JPEG,What is the object shown in this image used for,coloring the lips,Lipstick is for coloring the lips.,cutting paper; brushing teeth; coloring the li...,(A) cutting paper\n(B) brushing teeth\n(C) col...,ILSVRC,test,test_444,The image shows a lipstick. Lipsticks are typi...
2,3,data/images/fvqa/ILSVRC2012_test_00000444.JPEG,Where can you find the object in this image,a makeup cabinet,You are likely to find lipstick in a makeup ca...,a garden shed; a shoe rack; a makeup cabinet; ...,(A) a garden shed\n(B) a shoe rack\n(C) a make...,ILSVRC,test,test_444,The image shows lipstick. Lipstick is a cosmet...
3,4,data/images/fvqa/COCO_val2014_000000001584.jpg,Which object in this image can carry person?,bus,A bus is used to carry people.,tree; bicycle; bus; rock,(A) tree\n(B) bicycle\n(C) bus\n(D) rock,COCO,val,val_1584,The image shows a bus. A bus is a large vehicl...
4,5,data/images/fvqa/COCO_val2014_000000106920.jpg,Which food in this image is designed in Italian ?,pizza,pizza belongs to the category of Italian design.,sushi; pasta; pizza; tacos,(A) sushi\n(B) pasta\n(C) pizza\n(D) tacos,COCO,val,val_106920,The image shows a pizza. Pizza is a type of fo...
...,...,...,...,...,...,...,...,...,...,...,...
5821,5822,data/images/fvqa/ILSVRC2012_test_00007769.JPEG,which object in this image is capable of board...,a person,A person can board a train.,a bicycle; a tree; a person; a suitcase,(A) a bicycle\n(B) a tree\n(C) a person\n(D) a...,ILSVRC,test,test_7769,The image shows a person and a train. A person...
5822,5823,data/images/fvqa/ILSVRC2012_test_00002369.JPEG,What in the image is often encountered in the ...,turtles,You are likely to find Turtles in an ocean.,trees; rocks; turtles; bicycles,(A) trees\n(B) rocks\n(C) turtles\n(D) bicycles,ILSVRC,test,test_2369,The image shows turtles. Turtles are marine an...
5823,5824,data/images/fvqa/ILSVRC2012_test_00002369.JPEG,Which object in the image has a hard shell?,turtle,turtle has a shell.,feather; rock; turtle; sponge,(A) feather\n(B) rock\n(C) turtle\n(D) sponge,ILSVRC,test,test_2369,The image shows a turtle. The turtle has a she...
5824,5825,data/images/fvqa/ILSVRC2012_test_00007769.JPEG,which object in this image is capable of belie...,a person,A person can believe in Santa Claus.,a chair; a rock; a person; a tree,(A) a chair\n(B) a rock\n(C) a person\n(D) a tree,ILSVRC,test,test_7769,The image shows a person. A person is capable ...


# Get edit_ds by eval

In [ ]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)

# model
model = VQAModel(config)

# dataset
ds = VQADataset(config)
random.seed(getattr(config, "seed", 0))
ds.data = random.sample(ds.data, 50)
ds.set_dataloader(shuffle_choices=True)
ds.task_generate(model)
print(ds.task_engineer.eval(ds))

In [ ]:
pred_res_dir = os.path.join("results", "test", "pred", f"{config.model.name}", f"{config.experiment.dataset_name}")
os.makedirs(pred_res_dir, exist_ok=True)
pred_out_path = os.path.join(pred_res_dir, f"{config.experiment.task}_{config.experiment.split}.json")
edit_ds = ds.get_edits()
edit_ds.snap(out_path=pred_out_path)


# edit

In [ ]:
# model = VQAModel(config)
# load the prediction set back
import json
pred_set = json.load(open(pred_out_path))
edit_ds = VQADataset(config)
edit_ds.data = pred_set
edit_ds = edit_ds.get_edits()
print(len(edit_ds.data))

In [ ]:
import copy
model_old = copy.deepcopy(model)
# model_old_weights = copy.deepcopy(model.model.state_dict())
edit_ds.data

In [ ]:
# minimal single-batch finetune step (ft editor, no history)
editor = get_editor(config, model)
editor.generate = model.model.generate if hasattr(model, 'model') else model.generate
model.model.train()

batch = next(iter(edit_ds.loader))
tokens = model.prepare_training_batch(batch)
editor.edit(config, tokens, batch_history=None)

del tokens
torch.cuda.empty_cache()
# model_new_weights = copy.deepcopy(model.model.state_dict())
model_new = model

In [ ]:
edit_ds.task_generate(model_new)
edit_ds.data

In [ ]:
# model_old = copy.deepcopy(model)
# model_old.model.load_state_dict(model_old_weights)
edit_ds.task_generate(model_old)
edit_ds.data

## eval edits

### reliability

In [ ]:
from revlm.metrics import *

In [ ]:
reliability(model_old, edit_ds)

In [ ]:
reliability(model_new, edit_ds)

### generality

In [ ]:
related_texts = get_t_gen_input("fvqa", edit_ds)
related_images = get_i_gen_input("fvqa", edit_ds, k_per_model=2)
related_r_gen_df = get_r_gen_input("fvqa")

In [ ]:
# related_texts={}
# related_images={}
# for ex in edit_ds.data:
#     print(ex)
#     related_texts[ex['uid']] = [ex['question'], ex['question'], ex['question'], ex['question']]
#     related_images[ex['uid']] = [ex['image'], ex['image'], ex['image'], ex['image']]

print(image_generality(model_new, edit_ds, related_images))
print(text_generality(model_new, edit_ds, related_texts))
print(locality(model_old, model_new, edit_ds, sample_size=100))

In [ ]:
# 20m30s
editeval(model_old = model_old,
        model_new = model_new,
        edit_ds = edit_ds,
        editor = editor,
        related_texts = related_texts,
        related_images = related_images, 
        related_r_gen_df = related_r_gen_df,
        loc_sample_size = 100)

after runing revlm/run/r_gen_image.py

check the completeness of images

In [ ]:
# import os
# from PIL import Image, UnidentifiedImageError

# r_gen_d = get_r_gen_input("fvqa")
# remaining_sid = []
# remaining_uid = []

# for _, row in r_gen_d.iterrows():
#     image_path = row["image_path"]
#     try:
#         with Image.open(image_path) as img:
#             img.verify()  # ensure the file is a valid image
#     except (FileNotFoundError, UnidentifiedImageError, OSError) as exc:
#         print(f"Image load failed ({exc.__class__.__name__}): {image_path}")
#         remaining_sid.append(row["sid"])
#         remaining_uid.append(row["uid"])

In [ ]:
import os
from PIL import Image, UnidentifiedImageError

r_gen_d = get_r_gen_input("aokvqa")
remaining_sid = []
remaining_uid = []

bad = []

for _, row in r_gen_d.iterrows():
    path = row["image_path"]
    try:
        with Image.open(path) as img:
            img.verify()
    except (FileNotFoundError, UnidentifiedImageError, OSError, SyntaxError) as exc:
        print(f"Image load failed ({exc.__class__.__name__}): {path}")
        bad.append((row["sid"], row["uid"], path))

In [ ]:
# Check for bad images (including truncated) by actually loading and converting them
# This mimics what image_collate does: Image.open(path).convert("RGB")
import os
import struct
from PIL import Image, UnidentifiedImageError

def check_image_full_load(path):
    """Check if image can be fully loaded and converted to RGB (catches truncation errors)"""
    try:
        with Image.open(path) as img:
            # Actually convert to RGB - this is where truncation errors occur
            img_rgb = img.convert("RGB")
            # Access size to ensure image is fully decoded
            _ = img_rgb.size
        return True, None
    except (FileNotFoundError, UnidentifiedImageError, OSError, SyntaxError, struct.error) as exc:
        return False, exc

# Check aokvqa images (dedupe paths so each file is only loaded once)
r_gen_d = get_r_gen_input("aokvqa")
unique_paths = sorted(set(r_gen_d["image_path"].tolist()))
path_to_rows = {}
for idx, row in r_gen_d.iterrows():
    path = row["image_path"]
    path_to_rows.setdefault(path, []).append((row["sid"], row["uid"]))

bad_aokvqa = []
for i, path in enumerate(unique_paths):
    is_valid, exc = check_image_full_load(path)
    if not is_valid:
        print(f"Bad image ({exc.__class__.__name__}): {path}")
        # record all (sid, uid) that used this bad path
        for sid, uid in path_to_rows.get(path, []):
            bad_aokvqa.append((sid, uid, path, str(exc)))
    if (i + 1) % 1000 == 0:
        print(f"Checked {i+1}/{len(unique_paths)} images")

print(f"\nFound {len(bad_aokvqa)} bad images in aokvqa")
bad_aokvqa


In [ ]:
# import shutil
# from pathlib import Path

# src_dst = [#("./data/r_gen/image/fvqa/2018_2.png"), ("./data/r_gen/image/fvqa/2018_3.png"),
# ('./data/r_gen/image/aokvqa/5960_8.png', './data/r_gen/image/aokvqa/5960_7.png'),
# ('./data/r_gen/image/aokvqa/8710_4.png', './data/r_gen/image/aokvqa/8710_2.png')
# ]
# for src, dst in src_dst:
#     src = Path(src)
#     dst = Path(dst)
#     shutil.copy(src, dst)
#     print(f"Copied {src.name} -> {dst.name}")

In [ ]:
r_gen_d = get_r_gen_input("fvqa")
remaining_sid= []
remaining_uid= []
for _, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])



In [ ]:
r_gen_d[r_gen_d['sid'] == '2018_3']

In [ ]:
r_gen_d = get_r_gen_input("aokvqa")
remaining_sid= []
remaining_uid= []
remaining_rid= []
for rid, row in r_gen_d.iterrows():
    image_path = row["image_path"]
    if not os.path.exists(image_path):
        print(f"Image does not exist: {image_path}")
        remaining_sid.append(row["sid"])
        remaining_uid.append(row["uid"])
        remaining_rid.append(rid)



In [ ]:
# edit_ds.data

# unrelated_ds.df2data(pool_df)


In [ ]:
# # Test rationale_generality on a tiny dummy edit set
# import copy
# from revlm.metrics import rationale_generality

# # Require existing model/config/edit_ds from earlier cells
# try:
#     model  # noqa: F401
#     config  # noqa: F401
#     edit_ds  # noqa: F401
# except NameError:
#     raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# # Build a 2-sample dummy edit set from current edit_ds
# _dummy = copy.deepcopy(edit_ds)
# _dummy.data = _dummy.data[:2]
# _dummy.set_dataloader(shuffle_choices=False)

# # Map each uid to the other's uid to form a simple related_rationale
# uids = [_dummy.data[i]["uid"] for i in range(len(_dummy.data))]
# related_rationale = {}
# if len(uids) >= 2:
#     related_rationale = {uids[0]: [uids[1]], uids[1]: [uids[0]]}
# else:
#     # If only one example exists, just point to itself (degenerate case)
#     related_rationale = {uids[0]: [uids[0]]}

# print("rationale_generality:", rationale_generality(model, _dummy, related_rationale))


In [ ]:
# Test edit1_generality on the same tiny dummy edit set
import copy
from revlm.editors import get_editor
from revlm.metrics import edit1_generality, editk_boot_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build dummy edit set (reuse 2 examples)
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
# Keep evaluation deterministic and light
_dummy.config.n_iter = 1  # single training step inside editor.edit
_dummy.set_dataloader(shuffle_choices=True)

# Fresh base model for editing
model_old = copy.deepcopy(model)
editor = get_editor(config, model_old)
editor.generate = model_old.model.generate if hasattr(model_old, 'model') else model_old.generate

print("edit1_generality:", edit1_generality(model_old, _dummy, editor))

print("editk_boot_generality:", editk_boot_generality(model_old, _dummy, editor))
